# Tutorial: Filter and convert the X-Atlas/Orion dataset to AnnData

This notebook demonstrates how to generate an AnnData object for cells containing perturbation(s) of interest from the X-Atlas/Orion dataset.

## Import packages

In [1]:
import warnings
warnings.filterwarnings('ignore')

import anndata
import pandas as pd
import datasets 
from datasets import load_dataset
from scipy.sparse import csr_matrix

## Load data

### Dataset

Here we download the entire HCT116 dataset without streaming it. Once the dataset is downloaded, the dataset will be stored in the huggingface_hub cache, which avoids re-downloading the dataset files from Hugging Face every time they are used.

In [2]:
# download hct116 dataset
hct116_ds = load_dataset('Xaira-Therapeutics/X-Atlas-Orion', streaming=False, split='train', data_files='data/HCT116*.parquet')
hct116_ds

Dataset({
    features: ['gene_token_id', 'gene_expression', 'cell_barcode', 'sample', 'num_features', 'guide_target', 'gene_target', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'pass_guide_filter'],
    num_rows: 3409169
})

### Gene metadata

Gene metadata contains gene token ids and their corresponding ensembl IDs and official gene symbols.

In [3]:
gene_metadata = load_dataset('Xaira-Therapeutics/X-Atlas-Orion', name='gene_metadata', split='train')
gene_metadata

Dataset({
    features: ['ensembl_id', 'gene_name', 'gene_token_id'],
    num_rows: 38606
})

In [4]:
# read gene metadata into dataframe
gene_metadata_df = gene_metadata.to_pandas()

# set ensembl_id as index
gene_metadata_df = gene_metadata_df.set_index('ensembl_id')

gene_metadata_df

,gene_name,gene_token_id
ensembl_id,,
ENSG00000290825,DDX11L2,0
ENSG00000243485,MIR1302-2HG,1
ENSG00000237613,FAM138A,2
ENSG00000290826,ENSG00000290826,3
ENSG00000186092,OR4F5,4
...,...,...
ENSG00000277836,ENSG00000277836,38601
ENSG00000278633,ENSG00000278633,38602
ENSG00000276017,ENSG00000276017,38603


## Filter and convert dataset to anndata

### Filter for cells containing perturbations of interest

In [5]:
# specify which perturbations to include in filtered dataset
perturbation_list = ['TP53', 'CDKN1A']

In [6]:
# filter for cells containing these perturbations
sub_ds = hct116_ds.filter(
    lambda gene_target: gene_target in perturbation_list, 
    input_columns=['gene_target']
)
sub_ds

Dataset({
    features: ['gene_token_id', 'gene_expression', 'cell_barcode', 'sample', 'num_features', 'guide_target', 'gene_target', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'pass_guide_filter'],
    num_rows: 541
})

In [7]:
# filter for cells containing non-targeting controls (NTC)
ntc_ds = hct116_ds.filter(
    lambda gene_target: gene_target == 'Non-Targeting',
    input_columns=['gene_target']
)
ntc_ds

Dataset({
    features: ['gene_token_id', 'gene_expression', 'cell_barcode', 'sample', 'num_features', 'guide_target', 'gene_target', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'pass_guide_filter'],
    num_rows: 165777
})

Next, we will subsample the NTC cells to improve processing time.

In [8]:
# subsample ntcs to 500 randomly selected ntc cells
ntc_ds_sampled = ntc_ds.shuffle(seed=0).take(500)
ntc_ds_sampled

Dataset({
    features: ['gene_token_id', 'gene_expression', 'cell_barcode', 'sample', 'num_features', 'guide_target', 'gene_target', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'pass_guide_filter'],
    num_rows: 500
})

In [9]:
# if you want to filter for cells in a specific batch
ntc_ds_batch1 = hct116_ds.filter(
    lambda gene_target, sample: gene_target == 'Non-Targeting' and sample == 'HCT116_Batch1',
    input_columns=['gene_target', 'sample']
)
ntc_ds_batch1

Dataset({
    features: ['gene_token_id', 'gene_expression', 'cell_barcode', 'sample', 'num_features', 'guide_target', 'gene_target', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'pass_guide_filter'],
    num_rows: 891
})

## Convert dataset to anndata

In [10]:
def dataset_to_anndata(ds: datasets.arrow_dataset.Dataset, 
                       gene_dict: dict) -> anndata.AnnData:
    """
    Convert Dataset to AnnData format.
    
    Inputs:
        ds : datasets.arrow_dataset.Dataset
            Dataset containing 'gene_token_id', 'gene_expression', and 'cell_barcode' columns 
            plus metadata
        gene_dict : dict
            Dictionary mapping gene token IDs (int) to Ensembl gene names (str)
    
    Outputs:
        anndata.AnnData
            AnnData object with sparse expression matrix
    """
    
    # set-up gene mapping
    gene_token_ids = sorted(gene_dict.keys())
    token_to_col = {token: idx for idx, token in enumerate(gene_token_ids)}
    gene_names = [gene_dict[token] for token in gene_token_ids]
    
    # pre-allocate for CSR matrix components
    data, indices, ptr = [], [], [0]
    obs_records = []
    
    for i, row in enumerate(ds):
        genes = row['gene_token_id']
        expressions = row['gene_expression']
        
        # map to column indices
        gene_indices = [token_to_col[gene] for gene in genes]
        
        # add to CSR components
        data.extend(expressions)
        indices.extend(gene_indices)
        ptr.append(len(data))
        
        # extract metadata (exclude gene expression columns)
        metadata = {k: v for k, v in row.items() 
                   if k not in ['gene_token_id', 'gene_expression']}
        obs_records.append(metadata)
    
    # create sparse matrix
    n_cells = len(ptr) - 1
    n_genes = len(gene_names)

    csr_matrix_obj = csr_matrix((data, indices, ptr), shape=(n_cells, n_genes))

    # create AnnData
    obs_df = pd.DataFrame(obs_records)
    obs_df = obs_df.set_index('cell_barcode')  # set cell barcodes as index
    adata = anndata.AnnData(X=csr_matrix_obj, obs=obs_df)
    adata.var.index = pd.Index(gene_names, name='ensembl_id')
    
    return adata

In [11]:
gene_dict = {int(gene["gene_token_id"]): gene["ensembl_id"] for gene in gene_metadata}

In [12]:
# convert to anndata
adata = dataset_to_anndata(sub_ds, gene_dict)
adata.obs

,sample,num_features,guide_target,gene_target,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,pass_guide_filter
cell_barcode,,,,,,,,,
ACGTGCCCACTATCTT-HCT116_Batch1,HCT116_Batch1,2,CDKN1A_P1P2-1|CDKN1A_P1P2-2,CDKN1A,6361,27099.0,569.0,2.099709,1
CATAACGTCATAACAC-HCT116_Batch1,HCT116_Batch1,2,TP53_P1P2-1|TP53_P1P2-2,TP53,6786,32134.0,1022.0,3.180432,1
GCCTCGAAGGTATTCG-HCT116_Batch1,HCT116_Batch1,2,TP53_P1P2-1|TP53_P1P2-2,TP53,6206,26730.0,931.0,3.482978,1
AGTATTCAGGTCCAAC-HCT116_Batch10,HCT116_Batch10,2,CDKN1A_P1P2-1|CDKN1A_P1P2-2,CDKN1A,7939,40924.0,1991.0,4.865116,1
CAAATTCAGAAAGGTC-HCT116_Batch10,HCT116_Batch10,2,CDKN1A_P1P2-1|CDKN1A_P1P2-2,CDKN1A,5548,20248.0,969.0,4.785658,1
...,...,...,...,...,...,...,...,...,...
ATCCTTATCATGCTTC-HCT116_Batch98,HCT116_Batch98,2,CDKN1A_P1P2-1|CDKN1A_P1P2-2,CDKN1A,5214,14396.0,1238.0,8.599611,1
CCTGCACTCAAGCTAT-HCT116_Batch98,HCT116_Batch98,2,TP53_P1P2-1|TP53_P1P2-2,TP53,6189,17722.0,1155.0,6.517323,1
CGCCTACTCCCTTTGG-HCT116_Batch98,HCT116_Batch98,2,TP53_P1P2-1|TP53_P1P2-2,TP53,5528,18890.0,332.0,1.757544,1


In [13]:
# make ntc adata
ntc_adata = dataset_to_anndata(ntc_ds_sampled, gene_dict)
ntc_adata.obs

,sample,num_features,guide_target,gene_target,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,pass_guide_filter
cell_barcode,,,,,,,,,
ATGCCAGGTAAAGTGG-HCT116_Batch41,HCT116_Batch41,2,non-targeting_02178|non-targeting_03326,Non-Targeting,4154,11348.0,556.0,4.899542,1
AGCAGATAGCCTTAAG-HCT116_Batch63,HCT116_Batch63,2,non-targeting_00839|non-targeting_00103,Non-Targeting,5881,26404.0,1101.0,4.169823,1
CACTTACTCATACTAA-HCT116_Batch65,HCT116_Batch65,2,non-targeting_00364|non-targeting_01292,Non-Targeting,5692,20194.0,254.0,1.257799,1
CAAGCGAAGATCGACC-HCT116_Batch31,HCT116_Batch31,2,non-targeting_00642|non-targeting_00748,Non-Targeting,3862,9197.0,815.0,8.861585,1
GTCGAGTTCCCCTAGC-HCT116_Batch3,HCT116_Batch3,2,non-targeting_00599|non-targeting_03472,Non-Targeting,8105,51148.0,1448.0,2.831000,1
...,...,...,...,...,...,...,...,...,...
AGAAGGAAGAAATCAG-HCT116_Batch80,HCT116_Batch80,2,non-targeting_03005|non-targeting_02056,Non-Targeting,5174,20904.0,675.0,3.229047,1
AGACTTTAGATGTTCG-HCT116_Batch25,HCT116_Batch25,2,non-targeting_02097|non-targeting_02279,Non-Targeting,3852,12595.0,401.0,3.183803,1
CTCCTCAAGCGAAACG-HCT116_Batch49,HCT116_Batch49,2,non-targeting_01296|non-targeting_00786,Non-Targeting,5193,19820.0,388.0,1.957619,1
